In [14]:
# !pip install langgraph 
# !pip install langgraph-supervisor 
# !pip install langchain-community

In [ ]:
# 1. create tools 
# 2. create agents 
# 3. connect agents with tools 
# 4. define relationships 

In [13]:
# 1. create tool 
def sum(a:float,b:float)->float:
    """add two number"""
    return a + b 

def multiply(a:float,b:float)->float:
    """Multipy two numbers."""
    return a  * b 

In [14]:
# 3.  create websearch tool "duck duck go" 
# !pip  install ddgs
from  ddgs import DDGS
def research_tool(query:str)->str:
    """tool to research on internet, it return information from the internet, extract the web content"""
    text = "Title\n"
    with DDGS() as dd :
        response = dd.text(query)
        # print(len(response))
        # print(response)
        text += response[0]['title']
        text += "\n"
        text += response[0]['body']
    return text 

 

In [32]:
print(research_tool("who is prime minister of india?"))

Title
List ofprimeministersofIndia- Wikipedia
TheprimeministerofIndiais the chief executive of the GovernmentofIndiaand chair of the Union Council ofMinisters.


In [7]:
# !pip install langchain-openai

In [8]:
from langchain_openai import ChatOpenAI 
from dotenv import load_dotenv 
load_dotenv()

True

In [9]:
model = ChatOpenAI(model="gpt-4o-mini")

In [11]:
response = model.invoke("just tell me the name of prime minister of india? ")
print(response.content)

The Prime Minister of India is Narendra Modi.


In [2]:
from langgraph.prebuilt import create_react_agent

In [1]:
from langgraph_supervisor import create_supervisor 

In [15]:
math_agent = create_react_agent( 
    model=model, 
    tools=[sum,multiply],
    name="math_expert",
    prompt="you are a math expert. always use one tool at a time."   
)

C:\Users\Ranjit\AppData\Local\Temp\ipykernel_13700\195153151.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  math_agent = create_react_agent(


In [16]:
reserch_agent = create_react_agent(
    model=model,
    tools=[research_tool],
    name="research_agent",
    prompt="you are a world class researcher with a access to web search"

)

C:\Users\Ranjit\AppData\Local\Temp\ipykernel_13700\3711566020.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  reserch_agent = create_react_agent(


In [20]:
supervisor_agent_workflow = create_supervisor(
    model=model,
    agents=[math_agent,reserch_agent],
    prompt=(
        "You are a team supervisor managing a research expert and a math expert. "
        "For current events, use research_agent. "
        "For math problems, use math_agent."
    )
)

In [22]:
app = supervisor_agent_workflow.compile()

In [23]:
#### its time to test the agentic ai aplication 

query = input("Hey plz ask your question : ")
response = app.invoke({
    "messages":[
        {
            "role":"user",
            "content":query
        }
    ]
})

In [24]:
print(response)

{'messages': [HumanMessage(content='tell me the name of india prime minister?', additional_kwargs={}, response_metadata={}, id='0b6d999d-77c3-4cfe-9073-94f261a8d882'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_o88vKbFYh5Oq64ezqi06chvN', 'function': {'arguments': '{}', 'name': 'transfer_to_research_agent'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 103, 'total_tokens': 116, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_5714259854', 'id': 'chatcmpl-DO3H5fF0z6H8ud4w96y1uPLJUWSV6', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, name='supervisor', id='lc_run--019d2fe1-f1f3-7502-8eed-888d465ddf5e-0', tool_calls=[{'name': 'transfer_to_resear

In [25]:
for m in response["messages"]:
    m.pretty_print()

================================ Human Message =================================

tell me the name of india prime minister?
================================== Ai Message ==================================
Name: supervisor
Tool Calls:
  transfer_to_research_agent (call_o88vKbFYh5Oq64ezqi06chvN)
 Call ID: call_o88vKbFYh5Oq64ezqi06chvN
  Args:
================================= Tool Message =================================
Name: transfer_to_research_agent

Successfully transferred to research_agent
================================== Ai Message ==================================
Name: research_agent

The current Prime Minister of India is Narendra Modi.
================================== Ai Message ==================================
Name: research_agent

Transferring back to supervisor
Tool Calls:
  transfer_back_to_supervisor (032a8376-1980-4113-a5a0-201cfc0ce218)
 Call ID: 032a8376-1980-4113-a5a0-201cfc0ce218
  Args:
================================= Tool Message ========================

In [26]:
#### its time to test the agentic ai aplication 

query = input("Hey plz ask your question : ")
response = app.invoke({
    "messages":[
        {
            "role":"user",
            "content":query
        }
    ]
})

In [27]:
for m in response["messages"]:
    m.pretty_print()

================================ Human Message =================================

what is current weather of delhi in india, sum wit 2 and then multipy by the 2.
================================== Ai Message ==================================
Name: supervisor
Tool Calls:
  transfer_to_research_agent (call_OODRo3pKT6lsWa2pqjQtySyA)
 Call ID: call_OODRo3pKT6lsWa2pqjQtySyA
  Args:
================================= Tool Message =================================
Name: transfer_to_research_agent

Successfully transferred to research_agent
================================== Ai Message ==================================
Name: research_agent

The current weather in Delhi, India, cannot be directly extracted from the search results provided. However, you can check a reliable weather website like AccuWeather for detailed information.

Regarding your mathematical query: the sum of 2 and 2 is 4, and then multiplying that by 2 gives you \( 4 \times 2 = 8 \).
================================== Ai Mes

In [18]:
tpl = (
        "You are a team supervisor managing a research expert and a math expert. "
        "For current events, use research_agent. "
        "For math problems, use math_agent."
    )
print(tpl)
print(type(tpl))

You are a team supervisor managing a research expert and a math expert. For current events, use research_agent. For math problems, use math_agent.
<class 'str'>


In [17]:
tpl = (25,36,45,85,45) 
print(tpl)
print(type(tpl))

(25, 36, 45, 85, 45)
<class 'tuple'>
